# MQTT 實作：Publisher（發布端）

<a href="https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Publisher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

|[簡報](https://hackmd.io/@wiimax/ict_mqtt#/)|[Publisher](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Publisher.ipynb)|[Subscriber](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Subscriber.ipynb)|
|:-:|:-:|:-:|
|![](https://i.imgur.com/TFkKeuO.png)|![](https://i.imgur.com/uJdlGqo.png)|![](https://i.imgur.com/WVlVek0.png)|

## MQTT 實作

- 需要 MQTT 中間的 **Broker**：本範例使用公用的 `broker.emqx.io`
- 需要發布訊息的 **Publisher**：本筆記本
- 需要接收訊息的 **Subscriber**：[Subscriber 筆記本](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Subscriber.ipynb)

> 建議開兩個瀏覽器分頁，一邊執行 Publisher、一邊執行 Subscriber。先啟動 Subscriber，再啟動 Publisher，就能看到訊息即時送達。
>
> 公用 Broker 不需要帳號密碼，任何人都能訂閱同一個主題，請不要傳送個人資料、密碼或金鑰。

#### 熱門的公共 MQTT 伺服器：

![](https://i.imgur.com/GboMca8.png)

## Publisher

每 5 秒產生一筆 18～32 度的模擬溫度，連同時間包成 JSON 發布到主題 `nutc/csie/2701/sensor01`。

In [ ]:
!pip install paho-mqtt

In [ ]:
%%writefile mqtt_pub.py
# Publisher：每隔幾秒發布一筆模擬的溫度資料
#
# 執行：python mqtt_pub.py
# 停止：Ctrl+C（Colab 請按儲存格左側的停止鍵）

import datetime
import json
import time
from random import randint

import paho.mqtt.client as mqtt

BROKER = "broker.emqx.io"  # 也可改成 "broker.hivemq.com"
PORT = 1883
TOPIC = "nutc/csie/2701/sensor01"
QOS = 0  # 0：最多一次；1：至少一次；2：剛好一次
INTERVAL = 5  # 每次發布間隔幾秒
COUNT = 0  # 要發布幾筆；0 表示一直發布到手動停止


def make_message(temperature, now=None):
    """把溫度與時間包成一個 dict，之後再轉成 JSON 字串送出。"""
    now = now or datetime.datetime.now()
    return {"Temperature": temperature, "Time": now.strftime("%m/%d %H:%M:%S")}


def main():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.connect(BROKER, PORT, keepalive=60)
    client.loop_start()  # 在背景處理網路收發

    sent = 0
    try:
        while COUNT == 0 or sent < COUNT:
            message = make_message(randint(18, 32))
            info = client.publish(TOPIC, json.dumps(message), qos=QOS)
            info.wait_for_publish()
            sent += 1
            print(f"topic: {TOPIC}, message: {message}")
            if COUNT == 0 or sent < COUNT:
                time.sleep(INTERVAL)
    except KeyboardInterrupt:
        print("已停止發布")
    finally:
        client.loop_stop()
        client.disconnect()


if __name__ == "__main__":
    main()

In [ ]:
!python mqtt_pub.py

程式會一直執行，要停止請按儲存格左側的停止鍵（或選單「執行階段 → 中斷執行」）。想只送固定筆數，可以把 `COUNT` 改成例如 `10`。

### 練習
- 修改 `TOPIC` 主題，改成 `"tcnr/class/你的座號/sensor01"`，並把 Subscriber 的 `TOPIC` 改成 `"tcnr/#"`。
- 修改 Broker 為 `broker.hivemq.com`，記得 Publisher 與 Subscriber 要同步修改。
- 修改訊息的 `QOS` 為 1：Broker 收到後會回覆確認，Publisher 沒收到確認就會重送，確保訊息「至少送達一次」（因此可能重複收到）。
- 此筆記本用 `%%writefile mqtt_pub.py` 把程式存成 `.py` 檔，再用 `!python mqtt_pub.py` 在命令列執行；試著在自己電腦的終端機執行同一支腳本。

---

## Subscriber（精簡版）

Colab 同一本筆記本一次只會執行一個儲存格。要在這裡執行，請先停止上方的程式；也可以把這段程式貼到另一個分頁執行。完整版請見 [Subscriber 筆記本](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/15.MQTT/MQTT_Demo_Subscriber.ipynb)。

In [ ]:
# Subscriber（精簡版）

import paho.mqtt.client as mqtt


def on_connect(client, userdata, flags, reason_code, properties):
    print(f"connected with result code {reason_code}")
    client.subscribe("nutc/#")


def on_message(client, userdata, msg):
    print(f"{msg.topic} {msg.payload.decode('utf-8', errors='replace')}")


client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message

broker = "broker.emqx.io"
client.connect(broker, 1883, keepalive=60)

client.loop_forever()